In [22]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler

import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

from torchinfo import summary
import optuna

In [2]:
Raw_Train_Data = pd.read_csv(r"C:\Users\Banwa\Desktop\IWT\CODING\Aresnal\ML Practice\fashion-mnist_train.csv")
Raw_Test_Data = pd.read_csv(r"C:\Users\Banwa\Desktop\IWT\CODING\Aresnal\ML Practice\fashion-mnist_test.csv")
print("Datasets Loaded..")

Datasets Loaded..


In [3]:
print(Raw_Train_Data.head())

   label  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  \
0      2       0       0       0       0       0       0       0       0   
1      9       0       0       0       0       0       0       0       0   
2      6       0       0       0       0       0       0       0       5   
3      0       0       0       0       1       2       0       0       0   
4      3       0       0       0       0       0       0       0       0   

   pixel9  ...  pixel775  pixel776  pixel777  pixel778  pixel779  pixel780  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0        30        43         0   
3       0  ...         3         0         0         0         0         1   
4       0  ...         0         0         0         0         0         0   

   pixel781  pixel782  pixel783  pixel784  
0         0         0         

In [4]:
print("TRAINING DATA:")
print(Raw_Train_Data.info())
print("\n")
print("TEST DATA:")
print(Raw_Test_Data.info())


TRAINING DATA:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 359.3 MB
None


TEST DATA:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 785 entries, label to pixel784
dtypes: int64(785)
memory usage: 59.9 MB
None


In [5]:
print(Raw_Train_Data.describe())

              label        pixel1        pixel2        pixel3        pixel4  \
count  60000.000000  60000.000000  60000.000000  60000.000000  60000.000000   
mean       4.500000      0.000900      0.006150      0.035333      0.101933   
std        2.872305      0.094689      0.271011      1.222324      2.452871   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        2.000000      0.000000      0.000000      0.000000      0.000000   
50%        4.500000      0.000000      0.000000      0.000000      0.000000   
75%        7.000000      0.000000      0.000000      0.000000      0.000000   
max        9.000000     16.000000     36.000000    226.000000    164.000000   

             pixel5        pixel6        pixel7        pixel8        pixel9  \
count  60000.000000  60000.000000  60000.000000  60000.000000  60000.000000   
mean       0.247967      0.411467      0.805767      2.198283      5.682000   
std        4.306912      5.836188      8.215169    

In [6]:
print(Raw_Test_Data.describe())

              label        pixel1        pixel2        pixel3        pixel4  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean       4.500000      0.000400      0.010300      0.052100      0.077000   
std        2.872425      0.024493      0.525187      2.494315      2.208882   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        2.000000      0.000000      0.000000      0.000000      0.000000   
50%        4.500000      0.000000      0.000000      0.000000      0.000000   
75%        7.000000      0.000000      0.000000      0.000000      0.000000   
max        9.000000      2.000000     45.000000    218.000000    185.000000   

             pixel5        pixel6        pixel7        pixel8        pixel9  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean       0.208600      0.349200      0.826700      2.321200      5.457800   
std        4.669183      5.657849      8.591731    

In [7]:
X_train = Raw_Train_Data.iloc[:, 1:]
Y_train = Raw_Train_Data.iloc[:, :1]

X_test = Raw_Test_Data.iloc[:, 1:]
Y_test = Raw_Test_Data.iloc[:, :1]

In [8]:
X_train.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,5,0,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,1,2,0,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
fashion_mnist_labels = {
    0: 'T-shirt/top', 
    1: 'Trouser', 
    2: 'Pullover', 
    3: 'Dress', 
    4: 'Coat',
    5: 'Sandal', 
    6: 'Shirt', 
    7: 'Sneaker', 
    8: 'Bag', 
    9: 'Ankle boot'
}

indices = list(fashion_mnist_labels.keys())
names = list(fashion_mnist_labels.values())
dummy_values = [1] * len(indices)  # Give them all equal length

fig = go.Figure(
        go.Bar(
            x=dummy_values,
            y=names,
            orientation='h',
            # Map the text to look like "Class 0: T-shirt/top"
            text=[f"Class {i}: {name}" for i, name in zip(indices, names)],
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(size=16, color='white', family='Arial Black'),
            marker=dict(
                color=indices,
                colorscale='Plasma', # 'Plasma', 'Viridis', or 'Magma' look incredible in dark mode
                line=dict(color='rgba(255, 255, 255, 0)', width=0)
            ),
    hoverinfo='none' # Turn off the annoying hover box for a cleaner look
))

# 3. The Grandmaster Styling (Dark Mode & Clean Edges)
fig.update_layout(
    title='Fashion MNIST: Label Mapping Matrix',
    title_font=dict(size=24, family='Arial', color='#E0E0E0'),
    title_x=0.5, # Center the title
    plot_bgcolor='#121212',  # Deep tech-gray background
    paper_bgcolor='#121212',
    # Nuke the X-axis completely (we don't need numbers here)
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    # Reverse Y-axis so Class 0 is at the top, hide grid lines
    yaxis=dict(
        autorange="reversed", 
        showgrid=False, 
        zeroline=False, 
        tickfont=dict(color='#A0A0A0', size=14)
    ),
    margin=dict(l=20, r=20, t=80, b=20),
    height=500
)

# 4. Render the masterpiece
fig.show()

In [10]:
X_train = X_train/255.0
X_test = X_test/255.0

In [11]:
X_train_tensor = t.as_tensor(np.array(X_train), dtype=t.float32)
X_test_tensor = t.as_tensor(np.array(X_test), dtype=t.float32)

Y_train_tensor = t.as_tensor(np.array(Y_train), dtype=t.float32)
Y_test_tensor = t.as_tensor(np.array(Y_test), dtype=t.float32)

In [12]:
print(f"X Train Shape: {X_train_tensor.shape} | Y Train Shape: {Y_train_tensor.shape}")

X Train Shape: torch.Size([60000, 784]) | Y Train Shape: torch.Size([60000, 1])


## PIPELINE

In [13]:
class data(Dataset):
    def __init__(self, features, labels):
        super().__init__()
        self.features = t.tensor(features, dtype=t.float32)
        self.labels = t.tensor(labels.flatten(), dtype=t.long)

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

In [14]:
TrainDataset = data(X_train_tensor ,Y_train_tensor)
TestDataset = data(X_test_tensor, Y_test_tensor)

TrainLoader = DataLoader(TrainDataset, batch_size=64, pin_memory=True, shuffle=True)
TestLoader = DataLoader(TestDataset, batch_size=64, pin_memory=True, shuffle=False)

print(f"Train Dataset Length: {len(TrainDataset)} | Test Dataset: {len(TestDataset)}")

Train Dataset Length: 60000 | Test Dataset: 10000


C:\Users\Banwa\AppData\Local\Temp\ipykernel_28040\420812996.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.features = t.tensor(features, dtype=t.float32)
C:\Users\Banwa\AppData\Local\Temp\ipykernel_28040\420812996.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = t.tensor(labels.flatten(), dtype=t.long)


In [15]:
class Trial_ANN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super().__init__()

        layers = []

        for i in range(num_hidden_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer

        layers.append(nn.Linear(neurons_per_layer,output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [16]:
#Objective Function
def obejctive(trial):
    # HYPERPARMETERS
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step = 8)
    epochs = trial.suggest_int("epochs", 10, 100, step=10)
    learning_rate  = trial.suggest_float("learning_rate", 1e-9, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16,32,64,128])
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "SGD", "RMSprop"])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)


    TrainLoader = DataLoader(TrainDataset, batch_size=batch_size, pin_memory=True, shuffle=True)
    TestLoader = DataLoader(TestDataset, batch_size=batch_size, pin_memory=True, shuffle=False)
    # Model init
    input_dim = 784
    output_dim = 10

    model = Trial_ANN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model = model.to('cuda')
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    if optimizer_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)




    for epoch in range(epochs):

        for batch_features, batch_lables in TrainLoader:
            batch_features, batch_lables = batch_features.to('cuda'), batch_lables.to('cuda')
            # clear gradients
            optimizer.zero_grad()
            # forward
            outputs = model(batch_features)

            # loss
            loss = criterion(outputs, batch_lables)

            # optim
            loss.backward()

            # update 
            optimizer.step()
            
            # Metrics:
            # predictions_class = outputs.argmax(dim=1)
            # accuracy = (predictions_class == batch_lables).float().mean() * 100
            # print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {accuracy:.2f}")
    model.eval() # Lock the brain
    
    Final_accuracy = 0
    for batch_features, batch_labels in TestLoader:
        batch_labels = batch_labels.to('cuda')
        batch_features = batch_features.to('cuda')
        
        with t.no_grad(): # Turn off gradient tracking for maximum hardware efficiency
            test_preds = model(batch_features)
            test_preds_class = test_preds.argmax(dim=1)
            Final_accuracy = (test_preds_class == batch_labels).float().mean() * 100
    
    # print(f"Final Test Accuracy: {Final_accuracy.item():.2f}%")
    
    return Final_accuracy.item()
    

In [17]:
# study = optuna.create_study(direction='maximize')
# study.optimize(obejctive, n_trials=10)

In [18]:
# print(f"Best Value: {study.best_value:.2f} Parameters: {study.best_params}")

Best Value: 100.00 Parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 88, 'epochs': 80, 'learning_rate': 1.2792057833098903e-05, 'dropout_rate': 0.1, 'batch_size': 64, 'optimizer': 'AdamW', 'weight_decay': 1.804839841600478e-05}

In [19]:
class ANN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(128,88),
            nn.BatchNorm1d(88),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(88,88),
            nn.BatchNorm1d(88),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(88,10)
        )

    def forward(self, x):
        return self.model(x)

In [20]:
learining_rate = 1.2792057833098903e-05
epochs = 80
model = ANN(X_train.shape[1]).to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learining_rate, weight_decay=1.804839841600478e-05)

for epoch in range(epochs):

    for batch_features, batch_lables in TrainLoader:
        batch_features, batch_lables = batch_features.to('cuda'), batch_lables.to('cuda')
        # clear gradients
        optimizer.zero_grad()
        # forward
        outputs = model(batch_features)

        # loss
        loss = criterion(outputs, batch_lables)

        # optim
        loss.backward()

        # update 
        optimizer.step()
        
        # Metrics:

        predictions_class = outputs.argmax(dim=1)
        accuracy = (predictions_class == batch_lables).float().mean() * 100
        print(f"Epoch: {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {accuracy:.2f}")


Epoch: 1 | Loss: 2.3963 | Accuracy: 9.38
Epoch: 1 | Loss: 2.3875 | Accuracy: 9.38
Epoch: 1 | Loss: 2.3513 | Accuracy: 9.38
Epoch: 1 | Loss: 2.4958 | Accuracy: 0.00
Epoch: 1 | Loss: 2.3913 | Accuracy: 3.12
Epoch: 1 | Loss: 2.4617 | Accuracy: 10.94
Epoch: 1 | Loss: 2.4861 | Accuracy: 4.69
Epoch: 1 | Loss: 2.3768 | Accuracy: 6.25
Epoch: 1 | Loss: 2.4600 | Accuracy: 3.12
Epoch: 1 | Loss: 2.5189 | Accuracy: 7.81
Epoch: 1 | Loss: 2.3646 | Accuracy: 14.06
Epoch: 1 | Loss: 2.4367 | Accuracy: 6.25
Epoch: 1 | Loss: 2.3964 | Accuracy: 7.81
Epoch: 1 | Loss: 2.4085 | Accuracy: 10.94
Epoch: 1 | Loss: 2.2900 | Accuracy: 9.38
Epoch: 1 | Loss: 2.4224 | Accuracy: 9.38
Epoch: 1 | Loss: 2.3733 | Accuracy: 9.38
Epoch: 1 | Loss: 2.3163 | Accuracy: 18.75
Epoch: 1 | Loss: 2.3746 | Accuracy: 9.38
Epoch: 1 | Loss: 2.3172 | Accuracy: 12.50
Epoch: 1 | Loss: 2.2485 | Accuracy: 17.19
Epoch: 1 | Loss: 2.4240 | Accuracy: 14.06
Epoch: 1 | Loss: 2.3179 | Accuracy: 10.94
Epoch: 1 | Loss: 2.3298 | Accuracy: 12.50
Epoch: 

In [28]:
model.eval() # Lock the brain

Test_accuracies = []
for batch_features, batch_labels in TestLoader:
    batch_labels = batch_labels.to('cuda')
    batch_features = batch_features.to('cuda')
    
    with t.no_grad(): # Turn off gradient tracking for maximum hardware efficiency
        test_preds = model(batch_features)
        test_preds_class = test_preds.argmax(dim=1)
        test_acc = (test_preds_class == batch_labels).float().mean() * 100
        Test_accuracies.append(round(test_acc.item(),2))
        print(f"Final Test Accuracy: {test_acc.item():.2f}%")

print("Model Evaluation Complete")
sum = 0
for x in Test_accuracies:
    sum+=x
print(f"Mean Accuracy: {sum/len(Test_accuracies):.2f}%")
model.train() # Unlock the brain

# Print the architecture summary
summary(model, input_size=(1, 784))

Final Test Accuracy: 89.06%
Final Test Accuracy: 85.94%
Final Test Accuracy: 90.62%
Final Test Accuracy: 92.19%
Final Test Accuracy: 82.81%
Final Test Accuracy: 93.75%
Final Test Accuracy: 87.50%
Final Test Accuracy: 95.31%
Final Test Accuracy: 90.62%
Final Test Accuracy: 96.88%
Final Test Accuracy: 90.62%
Final Test Accuracy: 92.19%
Final Test Accuracy: 90.62%
Final Test Accuracy: 90.62%
Final Test Accuracy: 87.50%
Final Test Accuracy: 89.06%
Final Test Accuracy: 95.31%
Final Test Accuracy: 85.94%
Final Test Accuracy: 85.94%
Final Test Accuracy: 95.31%
Final Test Accuracy: 75.00%
Final Test Accuracy: 82.81%
Final Test Accuracy: 90.62%
Final Test Accuracy: 96.88%
Final Test Accuracy: 93.75%
Final Test Accuracy: 85.94%
Final Test Accuracy: 85.94%
Final Test Accuracy: 87.50%
Final Test Accuracy: 93.75%
Final Test Accuracy: 84.38%
Final Test Accuracy: 89.06%
Final Test Accuracy: 87.50%
Final Test Accuracy: 92.19%
Final Test Accuracy: 84.38%
Final Test Accuracy: 87.50%
Final Test Accuracy:

Layer (type:depth-idx)                   Output Shape              Param #
ANN                                      [1, 10]                   --
├─Sequential: 1-1                        [1, 10]                   --
│    └─Linear: 2-1                       [1, 128]                  100,480
│    └─BatchNorm1d: 2-2                  [1, 128]                  256
│    └─ReLU: 2-3                         [1, 128]                  --
│    └─Dropout: 2-4                      [1, 128]                  --
│    └─Linear: 2-5                       [1, 88]                   11,352
│    └─BatchNorm1d: 2-6                  [1, 88]                   176
│    └─ReLU: 2-7                         [1, 88]                   --
│    └─Dropout: 2-8                      [1, 88]                   --
│    └─Linear: 2-9                       [1, 88]                   7,832
│    └─BatchNorm1d: 2-10                 [1, 88]                   176
│    └─ReLU: 2-11                        [1, 88]                   --
